Connected to arena-env (3.13.16) (Python 3.13.16)

In [ ]:
import os
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Literal
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # don't let JAX pre-grab all GPU memory
import einops
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch as t
import torch.nn as nn
import torch.optim as optim
import wandb
from IPython.display import HTML, display
from jaxtyping import Bool, Float, Int
from matplotlib.animation import FuncAnimation
from numpy.random import Generator
from torch import Tensor
from torch.distributions.categorical import Categorical
from torch.optim.optimizer import Optimizer
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# Make sure exercises are in the path
chapter = "chapter2_rl"
section = "part3_ppo"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_ppo.tests as tests
from part1_intro_to_rl.utils import set_global_seeds
from part3_ppo.utils import arg_help
from plotly_utils import plot_cartpole_obs_and_dones
from rl_utils import AtariEnvs, BraxEnvs, LiveVideo, log_grid_video, log_video_html, record_grid_video, record_brax_video
from gpu_env import GPUEnv, CartPole, CartDoublePendulum, MountainCar, Pendulum, GPUProbe, angle_normalize, get_episode_data_from_infos

Arr = np.ndarray

# The networks in this section are tiny, and if you train on the CPU with
# PyTorch's default of one thread per core, threads spend longer
# coordinating than computing. Adjust if needed.
t.set_num_threads(min(4, t.get_num_threads()))

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
ENV_DICT = {"atari": AtariEnvs, "mujoco": BraxEnvs, "classic-control": CartPole, "swing-up": CartDoublePendulum,
            "mountain-car": MountainCar, "pendulum": Pendulum, "probe": GPUProbe}
EnvType = Literal["atari", "mujoco", "classic-control", "swing-up", "mountain-car", "pendulum", "probe"]
BatchedEnvs = GPUEnv | AtariEnvs | BraxEnvs  # anything `ENV_DICT` can build; all batched & tensor-native
# The bonus training loops below (Atari / MuJoCo / swing-up) each take a few minutes. They're guarded
# by `if SLOW:` so a top-to-bottom run of this file trains only the fast (~15s) CartPole; flip this to
# True to actually run the bonus environments.
SLOW = False

In [ ]:
@dataclass
class PPOArgs:
    # Basic / global
    seed: int = 1
    env_id: str = "CartPole-v1"
    mode: EnvType = "classic-control"
    device: str = device  # envs, networks and replay memory all live here; "cpu" is fastest for CartPole

    # Wandb / logging
    use_wandb: bool = False
    video_log_freq: int | None = 50  # every N phases, render a 4x4 grid video of the latest rollout
                                     # (logged to wandb if use_wandb, shown inline if live_viz)
    live_viz: bool = True  # display the 4x4 grid video inline every `video_log_freq` phases (notebook only)
    overwrite_video: bool = True  # each live video replaces the previous one in the notebook output (False appends them)
    wandb_project_name: str = "PPOCartPole"
    wandb_entity: str | None = None

    # Duration of training. One phase = a rollout of num_envs * num_steps_per_rollout env steps, then
    # num_minibatches * batches_per_learning_phase gradient steps on it. CartPole is solved after ~20 phases
    # (~2 s on CPU, ~4 s on a GPU); more parallel envs make each phase's batch bigger but don't help here.
    total_phases: int = 100
    num_envs: int = 64
    num_steps_per_rollout: int = 64
    num_minibatches: int = 4
    batches_per_learning_phase: int = 4

    # Optimization hyperparameters (a high LR converges in seconds on CartPole)
    lr: float = 5e-3
    max_grad_norm: float = 0.5

    # RL hyperparameters
    gamma: float = 0.99

    # PPO-specific hyperparameters
    gae_lambda: float = 0.95
    clip_coef: float = 0.2
    ent_coef: float = 0.01
    vf_coef: float = 1.0

    def __post_init__(self):
        self.batch_size = self.num_steps_per_rollout * self.num_envs

        assert self.batch_size % self.num_minibatches == 0, "batch_size must be divisible by num_minibatches"
        self.minibatch_size = self.batch_size // self.num_minibatches
        self.total_timesteps = self.total_phases * self.batch_size
        self.total_training_steps = self.total_phases * self.batches_per_learning_phase * self.num_minibatches
        self.device = t.device(self.device)


args = PPOArgs(num_minibatches=2)  # changing this also changes minibatch_size and total_training_steps
arg_help(args)

,default value,description
arg,,
seed,1,seed of the experiment
env_id,'CartPole-v1',the id of the environment
mode,'classic-control',"can be one of the following: 'classic-control', 'atari', 'mujoco', 'swing-up', 'mountain-car', 'pendulum' or 'probe'"
device,device(type='cuda'),"device the batched envs, networks and replay memory live on: 'cpu', 'cuda' or 'mps'"
use_wandb,False,"if toggled, this experiment will be tracked with Weights and Biases"
video_log_freq,50,"if not None, render the collected rollout as a 4x4 grid video every this many phases (shown inline if live_viz, logged to wandb if use_wandb)"
live_viz,True,"if toggled, the 4x4 grid video is displayed inline in the notebook every video_log_freq phases"
overwrite_video,True,"if toggled, each live video replaces the previous one in the notebook output; otherwise they are appended"
wandb_project_name,'PPOCartPole',the name of this experiment (also used as the wandb project name)


In [ ]:
def layer_init(layer: nn.Linear, std=np.sqrt(2), bias_const=0.0):
    t.nn.init.orthogonal_(layer.weight, std)
    t.nn.init.constant_(layer.bias, bias_const)
    return layer


def get_actor_and_critic(
    envs: BatchedEnvs,
    mode: EnvType = "classic-control",
) -> tuple[nn.Module, nn.Module]:
    """
    Returns (actor, critic), the networks used for PPO, using one of 3 different architectures depending on `mode`.
    """
    assert mode in ENV_DICT

    # Observations are continuous: `obs_shape` is the shape of one observation (e.g. (4,) for CartPole,
    # (4, 84, 84) for Atari), and `obs_dim` is the number of floats in it once flattened (4 for CartPole).
    obs_shape = envs.single_observation_space.shape
    obs_dim = int(np.prod(obs_shape))
    action_space = envs.single_action_space

    if mode in ("classic-control", "mountain-car", "probe"):
        # mountain-car (Discrete(3)) and the probe envs are also discrete classic-control tasks, so
        # they reuse this network. Discrete actions: `num_actions = action_space.n` is how many there are.
        num_actions = action_space.n
        actor, critic = get_actor_and_critic_classic(obs_dim, num_actions)
    elif mode == "atari":
        num_actions = action_space.n
        actor, critic = get_actor_and_critic_atari(obs_shape, num_actions)  # you'll implement these later
    elif mode in ("mujoco", "swing-up", "pendulum"):
        # swing-up (cart + double-pendulum) and pendulum are continuous-action tasks, so they reuse
        # the MuJoCo Gaussian actor/critic. Continuous actions: `action_dim` is the length of the action vector.
        action_dim = int(np.prod(action_space.shape))
        actor, critic = get_actor_and_critic_mujoco(obs_dim, action_dim)  # you'll implement these later
    else:
        raise ValueError(f"Unknown mode {mode!r}")

    return actor.to(device), critic.to(device)


def get_actor_and_critic_classic(obs_dim: int, num_actions: int):
    """
    Returns (actor, critic) in the "classic-control" case, according to diagram above.
    """
    actor = nn.Sequential([
        layer_init(nn.Linear(obs_dim, 64)),
        nn.Tanh(),
        layer_init(nn.Linear(64, 64)),
        nn.Tanh(),
        layer_init(nn.Linear(64, num_actions), std=0.01)
    ])

    critic = nn.Sequential([
        layer_init(nn.Linear(obs_dim, 64)),
        nn.Tanh(),
        layer_init(nn.Linear(64, 64)),
        nn.Tanh(),
        layer_init(nn.Linear(64, 1), std=1)
    ])

    return (actor, critic)


tests.test_get_actor_and_critic(get_actor_and_critic, mode="classic-control")

TypeError: list is not a Module subclass

In [ ]:
def layer_init(layer: nn.Linear, std=np.sqrt(2), bias_const=0.0):
    t.nn.init.orthogonal_(layer.weight, std)
    t.nn.init.constant_(layer.bias, bias_const)
    return layer


def get_actor_and_critic(
    envs: BatchedEnvs,
    mode: EnvType = "classic-control",
) -> tuple[nn.Module, nn.Module]:
    """
    Returns (actor, critic), the networks used for PPO, using one of 3 different architectures depending on `mode`.
    """
    assert mode in ENV_DICT

    # Observations are continuous: `obs_shape` is the shape of one observation (e.g. (4,) for CartPole,
    # (4, 84, 84) for Atari), and `obs_dim` is the number of floats in it once flattened (4 for CartPole).
    obs_shape = envs.single_observation_space.shape
    obs_dim = int(np.prod(obs_shape))
    action_space = envs.single_action_space

    if mode in ("classic-control", "mountain-car", "probe"):
        # mountain-car (Discrete(3)) and the probe envs are also discrete classic-control tasks, so
        # they reuse this network. Discrete actions: `num_actions = action_space.n` is how many there are.
        num_actions = action_space.n
        actor, critic = get_actor_and_critic_classic(obs_dim, num_actions)
    elif mode == "atari":
        num_actions = action_space.n
        actor, critic = get_actor_and_critic_atari(obs_shape, num_actions)  # you'll implement these later
    elif mode in ("mujoco", "swing-up", "pendulum"):
        # swing-up (cart + double-pendulum) and pendulum are continuous-action tasks, so they reuse
        # the MuJoCo Gaussian actor/critic. Continuous actions: `action_dim` is the length of the action vector.
        action_dim = int(np.prod(action_space.shape))
        actor, critic = get_actor_and_critic_mujoco(obs_dim, action_dim)  # you'll implement these later
    else:
        raise ValueError(f"Unknown mode {mode!r}")

    return actor.to(device), critic.to(device)


def get_actor_and_critic_classic(obs_dim: int, num_actions: int):
    """
    Returns (actor, critic) in the "classic-control" case, according to diagram above.
    """
    actor = nn.Sequential(
        layer_init(nn.Linear(obs_dim, 64)),
        nn.Tanh(),
        layer_init(nn.Linear(64, 64)),
        nn.Tanh(),
        layer_init(nn.Linear(64, num_actions), std=0.01)
    )

    critic = nn.Sequential(
        layer_init(nn.Linear(obs_dim, 64)),
        nn.Tanh(),
        layer_init(nn.Linear(64, 64)),
        nn.Tanh(),
        layer_init(nn.Linear(64, 1), std=1)
    )

    return (actor, critic)


tests.test_get_actor_and_critic(get_actor_and_critic, mode="classic-control")

All tests in `test_get_actor_and_critic(mode='classic-control')` passed!


In [ ]:
@t.inference_mode()
def compute_advantages(
    next_value: Float[Tensor, "num_envs"],
    next_terminated: Bool[Tensor, "num_envs"],
    rewards: Float[Tensor, "buffer_size num_envs"],
    values: Float[Tensor, "buffer_size num_envs"],
    terminated: Bool[Tensor, "buffer_size num_envs"],
    gamma: float,
    gae_lambda: float,
) -> Float[Tensor, "buffer_size num_envs"]:
    """
    Compute advantages using Generalized Advantage Estimation.
    """
    raise NotImplementedError()


tests.test_compute_advantages(compute_advantages)

NotImplementedError: 